# Experiment 4: Test-Time Adaptation on LPN's Pattern-2D

Test-time adaptation of a pretrained **Latent Program Network (LPN)** (Bonnet & Macfarlane, "Searching Latent Program Spaces") on its **Pattern-2D** task. Compares three conditions on the same fixed set of tasks: `mean` (no adaptation), `gradient_ascent` (the paper's own search over the 2D latent vector), and `lora_ascent` (ours -- a per-task LoRA adapter fit on the decoder's weights instead of the latent).

This is a **separate JAX/Flax pipeline**, not the PyTorch/HuggingFace one experiments 1-3 use -- it does **not** build on `colab_bootstrap.ipynb` (different dependencies: JAX/Flax/Optax instead of `transformers`/`peft`, and the `clement-bonnet/lpn` repo itself). Run this notebook's own bootstrap cell below, in its own Colab session, rather than one that's already run experiments 1-3.

Config: `src/config/exp4_lpn_pattern2d.yaml`

In [ ]:
# 1. Mount Drive (for persistent output storage across ephemeral runtimes)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Get this repo's code
!git clone https://github.com/jtylerleake/LoRA-Experiments.git /content/lora_experiments
%cd /content/lora_experiments

In [ ]:
# 3. Get the lpn repo (JAX/Flax model + its own Pattern-2D task generator)
# and install its pinned deps on top of Colab's preinstalled jaxlib, plus
# this repo's own package. lpn isn't a properly installable package -- its
# own README has you add its clone dir to PYTHONPATH instead of `pip install
# -e`, since its code imports itself as `src.models...` (absolute, rooted at
# the repo root) -- so we do the same rather than fight that. Its
# requirements.txt pins jax/jaxlib==0.4.26 (mid-2024); unverified whether
# that's still compatible with Colab's current CUDA runtime -- if the pinned
# install fails, try dropping the jax/jaxlib pins and letting pip resolve
# against whatever Colab already has.
!git clone https://github.com/clement-bonnet/lpn.git /content/lpn
!pip install -q -r /content/lpn/requirements.txt
!pip install -q -e .

import sys
sys.path.insert(0, "/content/lpn")

In [ ]:
# 4. Run the experiment. Called in-process (not `!python ...`) so tqdm's
# progress bars are the only thing that render in this cell's output -- see
# scripts/run_experiment.py's notebooks for why (same lesson applies here).
import sys
sys.path.insert(0, "scripts")
from run_exp4 import main as run_exp4

run_exp4([
    "--config", "src/config/exp4_lpn_pattern2d.yaml",
    "--output-root", "/content/drive/MyDrive/lora_experiments_outputs",
])

In [ ]:
# 5. Plot the condition comparison (mean vs. gradient_ascent vs. lora_ascent).
!python scripts/plot_exp4_results.py \
    --metrics /content/drive/MyDrive/lora_experiments_outputs/exp4_lpn_pattern2d/metrics.jsonl \
    --output-dir /content/drive/MyDrive/lora_experiments_outputs/exp4_lpn_pattern2d/plots